**Часть A — программный.**

Используя BeautifulSoup, извлечь с разрешённой статической учебной страницы или предоставленного HTML-файла:
- название объекта;
- категорию;
- числовой показатель: цену, рейтинг, дату или количество;
- ссылку;
- идентификатор, если он доступен в `data-*`-атрибуте.

Требования:

- получить не менее 20 записей;
- сохранить результат в `CSV`;
- очистить пробелы, привести числовые поля к числовому типу;
- удалить дубликаты;
- вывести число строк, долю пропусков и первые пять записей;
- реализовать `timeout` и не менее двух повторных попыток запроса;
- кратко описать CSS-селекторы или правила поиска.

In [ ]:
import time
from requests.exceptions import RequestException
import pandas as pd
import requests
from bs4 import BeautifulSoup

def fetch_html(url, retries=3, timeout=15):
    headers = {"User-Agent": "Mozilla/5.0 (educational demo)"}
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=timeout)
            response.raise_for_status()
            return response.text
        except RequestException as error:
            print(f"Попытка {attempt}/{retries}: {error}")
            if attempt == retries:
                raise
            time.sleep(2 * attempt)

In [49]:
url = "https://books.toscrape.com/"

t1 = time.time()

pages = []
for i in range(1, 51):
    response = fetch_html(url=f'{url}catalogue/page-{i}.html')
    pages.append( BeautifulSoup(response, "lxml"))

products_list = []

for soup in pages:
    products = soup.select("article.product_pod")

    for product in products:
        title = product.select_one("h3 a")["title"]
        price = float(product.find("div", class_="product_price").select_one("p").get_text()[2:])
        link = f'{url}{product.select_one("h3 a")["href"]}'
        products_list.append({
            'title': title,
            'price': price,
            'url': link
        })
    
df = pd.DataFrame(products_list).drop_duplicates(subset=['url'])
df.to_csv('results/books.csv', index=False, encoding='utf-8-sig')

t2 = time.time() - t1


Попытка 1/3: HTTPSConnectionPool(host='books.toscrape.com', port=443): Max retries exceeded with url: /catalogue/page-22.html (Caused by ConnectTimeoutError(<HTTPSConnection(host='books.toscrape.com', port=443) at 0x1da830ed6d0>, 'Connection to books.toscrape.com timed out. (connect timeout=15)'))


In [51]:
print(f'Время работы: {t2:.2f}с.')

#print(df.shape())
df

Время работы: 76.82с.


,title,price,url
0,A Light in the Attic,51.77,https://books.toscrape.com/a-light-in-the-atti...
1,Tipping the Velvet,53.74,https://books.toscrape.com/tipping-the-velvet_...
2,Soumission,50.10,https://books.toscrape.com/soumission_998/inde...
3,Sharp Objects,47.82,https://books.toscrape.com/sharp-objects_997/i...
4,Sapiens: A Brief History of Humankind,54.23,https://books.toscrape.com/sapiens-a-brief-his...
...,...,...,...
995,Alice in Wonderland (Alice's Adventures in Won...,55.53,https://books.toscrape.com/alice-in-wonderland...
996,"Ajin: Demi-Human, Volume 1 (Ajin: Demi-Human #1)",57.06,https://books.toscrape.com/ajin-demi-human-vol...
997,A Spy's Devotion (The Regency Spies of London #1),16.97,https://books.toscrape.com/a-spys-devotion-the...
998,1st to Die (Women's Murder Club #1),53.98,https://books.toscrape.com/1st-to-die-womens-m...


**Часть B — no-code.**

Создать в Octoparse workflow для той же или аналогичной статической страницы:

- выбрать повторяющуюся запись;
- извлечь не менее четырёх полей;
- настроить пагинацию либо явно объяснить её отсутствие;
- выполнить тестовый запуск;
- экспортировать результат;
- приложить скриншоты workflow и таблицы результата;
- сравнить число записей и набор полей с программной реализацией.

![image.png](screenshots\image.png)

no-code вариант отработал за 3 c лишним минуты. Со всех страниц собрано 1000 записей, как и вариант через python.